[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# Cómo entrenar Clasificación YOLOv8 en un conjunto de datos personalizado

---

[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/how-to-train-yolov8-on-a-custom-dataset)
[![YouTube](https://badges.aleen42.com/src/youtube.svg)](https://youtu.be/wuZtUMEiKWY)
[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

Ultralytics YOLOv8 es la última versión del modelo de detección de objetos y segmentación de imágenes YOLO (You Only Look Once) desarrollado por Ultralytics. El modelo YOLOv8 está diseñado para ser rápido, preciso y fácil de usar, lo que lo convierte en una excelente opción para una amplia gama de tareas de detección de objetos y segmentación de imágenes. Puede entrenarse en grandes conjuntos de datos y es capaz de ejecutarse en una variedad de plataformas de hardware, desde CPU hasta GPU.

## ⚠️ Descargo de responsabilidad

A partir del 18.01.2023, la Clasificación YOLOv8 parece un poco subdesarrollada. Es posible entrenar modelos, pero su usabilidad es cuestionable. Los problemas conocidos incluyen:
- El modelo preentrenado en el conjunto de datos Imagenet opera sobre el ID de las clases y no sobre sus nombres. Solo después de un posprocesamiento personalizado se puede saber cómo se clasificó la imagen.
- No hay datos de entrenamiento detallados disponibles. En este punto, es casi un estándar guardar información como la matriz de confusión o gráficos de métricas clave después de completar una sesión de entrenamiento. YOLOv8 ofrece esta función pero por el momento solo para Detección de Objetos y Segmentación de Instancias.
- En el caso de la CLI, no se guardan las predicciones en forma de texto. Están anotadas en la imagen, lo que en la práctica imposibilita la construcción de cualquier aplicación que las utilice.
- Con SDK, se devuelve una matriz difícil de interpretar en lugar de un vector de probabilidades. Solo después de un posprocesamiento personalizado se puede saber cómo se clasificó la imagen.

## Publicación de blog adjunta

Le recomendamos que siga este cuaderno mientras lee la publicación del blog sobre cómo entrenar la Clasificación YOLOv8, al mismo tiempo.

## Consejo profesional: usar aceleración de GPU

Si está ejecutando este cuaderno en Google Colab, navegue hasta `Editar` -> `Configuración del cuaderno` -> `Acelerador de hardware`, configúrelo en `GPU` y luego haga clic en `Guardar`. Esto asegurará que su cuaderno use una GPU, lo que acelerará significativamente los tiempos de entrenamiento del modelo.

## Pasos en este Tutorial

En este tutorial, cubriremos:

- Antes de empezar
- Instalar YOLOv8
- Conceptos básicos de CLI
- Inferencia con Modelo COCO Preentrenado
- Roboflow Universe
- Preparación de un conjunto de datos personalizado
- Entrenamiento personalizado
- Validar modelo personalizado
- Inferencia con modelo personalizado

**¡Comencemos!**

## Antes de empezar

Asegurémonos de que tenemos acceso a la GPU. Podemos usar el comando `nvidia-smi` para hacer eso. En caso de cualquier problema, navegue hasta `Editar` -> `Configuración del cuaderno` -> `Acelerador de hardware`, configúrelo en `GPU` y luego haga clic en `Guardar`.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Instalar YOLOv8

Nos esforzamos para que nuestros cuadernos YOLOv8 funcionen con la última versión de la biblioteca. Las últimas pruebas se realizaron el **06.10.2024** con la versión **YOLOv8.3.7**.

Si nota que nuestro cuaderno se comporta incorrectamente, especialmente si experimenta errores que le impiden seguir el tutorial, ¡no lo dude! Háganoslo saber y abra un [problema](https://github.com/roboflow/notebooks/issues) en el repositorio de Cuadernos de Roboflow.

YOLOv8 se puede instalar de dos maneras: desde la fuente y a través de pip. Esto se debe a que es la primera iteración de YOLO en tener un paquete oficial.

In [ ]:
# Método de instalación Pip (recomendado)

!pip install ultralytics==8.2.103 -q
# evitar que ultralytics rastree tu actividad
!yolo settings sync=False
from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

In [ ]:
# Método de clonación de Git (para desarrollo)

# %cd {HOME}
# !git clone github.com/ultralytics/ultralytics
# %cd {HOME}/ultralytics
# !pip install -e .

# from IPython import display
# display.clear_output()

# import ultralytics
# ultralytics.checks()

In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

## Conceptos básicos de CLI

Si desea entrenar, validar o ejecutar inferencias en modelos y no necesita realizar modificaciones en el código, usar la interfaz de línea de comandos de YOLO es la forma más fácil de comenzar. Lea más sobre CLI en los [Documentos de Ultralytics YOLO](https://docs.ultralytics.com/usage/cli/).

```
yolo task=detect    mode=train    model=yolov8n.yaml      args...
          classify       predict        yolov8n-cls.yaml  args...
          segment        val            yolov8n-seg.yaml  args...
                         export         yolov8n.pt        format=onnx  args...
```

## Inferencia con Modelo COCO Preentrenado

### 💻 CLI

`yolo mode=predict` ejecuta la inferencia de YOLOv8 en una variedad de fuentes, descargando modelos automáticamente de la última versión de YOLOv8 y guardando los resultados en `runs/predict`.

In [ ]:
%cd {HOME}
!yolo task=classify mode=predict model=yolov8n-cls.pt conf=0.25 source='https://media.roboflow.com/notebooks/examples/dog.jpeg'

In [ ]:
%cd {HOME}
Image(filename='runs/classify/predict/dog.jpeg', height=600)

### 🐍 SDK de Python

La forma más sencilla de utilizar simplemente YOLOv8 directamente en un entorno de Python.

In [ ]:
model = YOLO(f'{HOME}/yolov8n-cls.pt')
results = model.predict(source='https://media.roboflow.com/notebooks/examples/dog.jpeg', conf=0.25)

## Roboflow Universe

¿Necesita datos para su proyecto? Antes de perder tiempo anotando, consulte Roboflow Universe, un repositorio de más de 110,000 conjuntos de datos de código abierto que puede usar en sus proyectos. Encontrará conjuntos de datos que contienen de todo, desde grietas anotadas en hormigón hasta imágenes de plantas con anotaciones de enfermedades.


[![Roboflow Universe](https://media.roboflow.com/notebooks/template/uni-banner-frame.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672878480290)](https://universe.roboflow.com/)



## Preparación de un conjunto de datos personalizado

Construir un conjunto de datos personalizado puede ser un proceso doloroso. Puede llevar docenas o incluso cientos de horas recopilar imágenes, etiquetarlas y exportarlas en el formato adecuado. Afortunadamente, Roboflow hace que este proceso sea lo más sencillo y rápido posible. ¡Déjame mostrarte cómo!

### Paso 1: Creación del proyecto

Antes de comenzar, debe crear una [cuenta](https://app.roboflow.com/login) de Roboflow. Una vez que lo haga, puede crear un nuevo proyecto en el [panel](https://app.roboflow.com/) de Roboflow. Tenga en cuenta elegir el tipo de proyecto correcto. En nuestro caso, Detección de Objetos.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/creating-project.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672929799852"
  >
</div>

### Paso 2: Subir imágenes

A continuación, agregue los datos a su proyecto recién creado. Puede hacerlo a través de API o mediante nuestra [interfaz web](https://docs.roboflow.com/adding-data/object-detection).

Si arrastra y suelta un directorio con un conjunto de datos en un formato compatible, el panel de Roboflow leerá automáticamente las imágenes y las anotaciones juntas.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/uploading-images.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672929808290"
  >
</div>

### Paso 3: Etiquetado

Si solo tiene imágenes, puede etiquetarlas en [Roboflow Annotate](https://docs.roboflow.com/annotate).

<div align="center">
  <img
    width="640"
    src="https://user-images.githubusercontent.com/26109316/210901980-04861efd-dfc0-4a01-9373-13a36b5e1df4.gif"
  >
</div>

### Paso 4: Generar nueva versión del conjunto de datos

Ahora que tenemos agregadas nuestras imágenes y anotaciones, podemos generar una Versión del Conjunto de Datos. Al generar una Versión, puede optar por agregar preprocesamiento y aumentos. Este paso es completamente opcional, sin embargo, puede permitirle mejorar significativamente la robustez de su modelo.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/generate-new-version.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1673003597834"
  >
</div>

### Paso 5: Exportar el conjunto de datos

Una vez generada la versión del conjunto de datos, tenemos un conjunto de datos alojado que podemos cargar directamente en nuestro cuaderno para facilitar el entrenamiento. Haga clic en `Export` y seleccione el formato de conjunto de datos `Folder Structure`.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/export.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672943313709"
  >
</div>




In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

!pip install roboflow --quiet

import roboflow

roboflow.login()

rf = roboflow.Roboflow()

project = rf.workspace("model-examples").project("banana-ripeness-classification-cdvrn")
dataset = project.version(1).download("folder")

In [ ]:
!mv {dataset.location}/valid {dataset.location}/val

## Entrenamiento personalizado

In [ ]:
%cd {HOME}

!yolo task=classify mode=train model=yolov8n-cls.pt data={dataset.location} epochs=25 imgsz=128

In [ ]:
!ls -la {HOME}/runs/classify/train/

In [ ]:
!cat {HOME}/runs/classify/train/results.csv | head -10

## Validar modelo personalizado

In [ ]:
%cd {HOME}

!yolo task=classify mode=val model={HOME}/runs/classify/train/weights/best.pt data={dataset.location}

## Inferencia con modelo personalizado

In [ ]:
%cd {HOME}
!yolo task=classify mode=predict model={HOME}/runs/classify/train/weights/best.pt conf=0.25 source={dataset.location}/test/overripe